In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

OUTPUT_DIR = "/content/drive/MyDrive/DL_OUTPUT_2026"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Saving everything to:", OUTPUT_DIR)

Mounted at /content/drive
Saving everything to: /content/drive/MyDrive/DL_OUTPUT_2026


In [2]:
from google.colab import files
uploaded = files.upload()

Saving dataset.zip to dataset.zip


In [5]:
import zipfile

with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/dataset/dataset")

base_dir = "/content/dataset/dataset/dataset"

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224,224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    base_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    base_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

Found 400 images belonging to 2 classes.
Found 100 images belonging to 2 classes.


In [7]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = train_data.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)

class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)

{0: np.float64(1.0), 1: np.float64(1.0)}


In [8]:
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras import layers, models

def build_vgg16():
    base = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))
    for l in base.layers: l.trainable = False
    x = layers.Flatten()(base.output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(base.input, out)

def build_resnet():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
    for l in base.layers: l.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(base.input, out)

def build_mobilenet():
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
    for l in base.layers: l.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(base.input, out)

In [9]:
from tensorflow.keras.optimizers import Adam
import json

def train_and_save(model, name):
    model.compile(
        optimizer=Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=10,
        class_weight=class_weight_dict
    )

    # Save model
    model.save(f"{OUTPUT_DIR}/{name}.h5")

    # Save history
    with open(f"{OUTPUT_DIR}/{name}_history.json", "w") as f:
        json.dump(history.history, f)

    return history

In [10]:
vgg_model = build_vgg16()
resnet_model = build_resnet()
mobilenet_model = build_mobilenet()

hist_vgg = train_and_save(vgg_model, "VGG16")
hist_resnet = train_and_save(resnet_model, "ResNet50")
hist_mobile = train_and_save(mobilenet_model, "MobileNetV2")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step - accuracy: 0.8775 - loss: 0.2838 - val_accuracy: 0.8800 - val_loss: 0.2202
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 690ms/step - accuracy: 0.9875 - loss: 0.0541 - val_accuracy: 0.9700 - val_loss: 0.0742
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 550ms/step - accuracy: 0.9975 - loss: 0.0191 - val_accuracy: 0.9300 - val_loss: 0.0994
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 659ms/step - accuracy: 1.0000 - loss: 0.0075 - val_accuracy: 0.9700 - val_loss: 0.0509
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 648ms/step - accuracy: 0.9975 - loss: 0.0069 - val_accuracy: 0.9800 - val_loss: 0.0387
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 561ms/step - accuracy: 1.0000 - loss: 0.0047 - val_accuracy: 0.9900 - val_loss: 0.0391
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 680ms/step - accuracy: 1.0000 - loss: 0.00

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 32s 2s/step - accuracy: 0.5375 - loss: 0.7333 - val_accuracy: 0.5000 - val_loss: 0.6737
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 518ms/step - accuracy: 0.5275 - loss: 0.6888 - val_accuracy: 0.9300 - val_loss: 0.6563
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 672ms/step - accuracy: 0.6150 - loss: 0.6642 - val_accuracy: 0.6900 - val_loss: 0.6503
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 511ms/step - accuracy: 0.7525 - loss: 0.6538 - val_accuracy: 0.9500 - val_loss: 0.6298
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 661ms/step - accuracy: 0.8125 - loss: 0.6482 - val_accuracy: 1.0000 - val_loss: 0.6197
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 519ms/step - accuracy: 0.8075 - loss: 0.6377 - val_accuracy: 0.9900 - val_loss: 0.6152
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 617ms/step - accuracy: 0.8150 - loss: 0.6252 - val_accuracy: 0.9400 - val_loss: 0.6077
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 553ms/step - accuracy: 0.7600 - loss: 0.6210 - val_accuracy: 0.91

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 53s 3s/step - accuracy: 0.4900 - loss: 0.7214 - val_accuracy: 0.6900 - val_loss: 0.5760
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 480ms/step - accuracy: 0.5625 - loss: 0.6785 - val_accuracy: 0.7700 - val_loss: 0.5493
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 633ms/step - accuracy: 0.6675 - loss: 0.6235 - val_accuracy: 0.8300 - val_loss: 0.4945
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 492ms/step - accuracy: 0.7550 - loss: 0.5883 - val_accuracy: 0.8300 - val_loss: 0.4868
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 622ms/step - accuracy: 0.7525 - loss: 0.5634 - val_accuracy: 0.8700 - val_loss: 0.4549
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 480ms/step - accuracy: 0.8225 - loss: 0.5125 - val_accuracy: 0.8900 - val_loss: 0.4437
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 637ms/step - accuracy: 0.8225 - loss: 0.4948 - val_accuracy: 0.9000 - val_loss: 0.4123
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 476ms/step - accuracy: 0.8750 - loss: 0.4556 - val_accuracy: 0.92

In [11]:
from sklearn.metrics import classification_report, roc_curve, auc
import matplotlib.pyplot as plt

def evaluate_and_save(model, name):
    val_data.reset()
    preds = model.predict(val_data)
    y_pred = (preds > 0.5).astype(int)
    y_true = val_data.classes

    # Classification report
    report = classification_report(y_true, y_pred)
    with open(f"{OUTPUT_DIR}/{name}_report.txt", "w") as f:
        f.write(report)

    # ROC
    fpr, tpr, _ = roc_curve(y_true, preds)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--')
    plt.title(f"{name} ROC Curve")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()
    plt.savefig(f"{OUTPUT_DIR}/{name}_ROC.png")
    plt.close()

    return fpr, tpr, roc_auc

In [12]:
fpr_vgg, tpr_vgg, auc_vgg = evaluate_and_save(vgg_model, "VGG16")
fpr_res, tpr_res, auc_res = evaluate_and_save(resnet_model, "ResNet50")
fpr_mob, tpr_mob, auc_mob = evaluate_and_save(mobilenet_model, "MobileNetV2")

4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 389ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step


3/4 ━━━━━━━━━━━━━━━━━━━━ 0s 700ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step


In [13]:
plt.figure()
plt.plot(fpr_vgg, tpr_vgg, label=f"VGG16 ({auc_vgg:.3f})")
plt.plot(fpr_res, tpr_res, label=f"ResNet50 ({auc_res:.3f})")
plt.plot(fpr_mob, tpr_mob, label=f"MobileNetV2 ({auc_mob:.3f})")
plt.plot([0,1],[0,1],'--')

plt.legend()
plt.title("ROC Comparison")
plt.xlabel("FPR")
plt.ylabel("TPR")

plt.savefig(f"{OUTPUT_DIR}/ROC_Comparison.png")
plt.close()

In [14]:
def save_plots(history, name):
    # Accuracy
    plt.figure()
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title(name + " Accuracy")
    plt.legend(['train','val'])
    plt.savefig(f"{OUTPUT_DIR}/{name}_accuracy.png")
    plt.close()

    # Loss
    plt.figure()
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title(name + " Loss")
    plt.legend(['train','val'])
    plt.savefig(f"{OUTPUT_DIR}/{name}_loss.png")
    plt.close()

save_plots(hist_vgg, "VGG16")
save_plots(hist_resnet, "ResNet50")
save_plots(hist_mobile, "MobileNetV2")

| Model       | Accuracy       | ROC-AUC        | Precision | Recall   | F1             |
| ----------- | -------------- | -------------- | --------- | -------- | -------------- |
| **VGG16**   | Highest        | Highest        | Best      | Balanced | Best           |
| ResNet50    | Slightly lower | Slightly lower | High      | High     | Slightly lower |
| MobileNetV2 | Lowest         | Lowest         | Moderate  | Moderate | Lowest         |
